# Predicting Laptop Prices — Extended Project
## Multi-Model Regression with Advanced Feature Engineering

**Skeleton Notebook** — instructions and structure only. Fill in every `# TODO` cell yourself.

### Project brief
You work for a price-intelligence startup. Retail partners want a model that predicts a laptop's fair market price from its hardware specs, so they can flag underpriced/overpriced listings. You are given `laptop_price.csv` (823 laptops, 19 raw columns) and must build, tune, and compare several regression models, ending with a defensible "best" model plus an explanation of what drives laptop prices.

This extended version goes further than a basic pipeline: it includes a full data-quality audit, domain-derived features, multicollinearity checks, a leakage-safe encoding strategy, several competing model families, hyperparameter search, cross-validation, two kinds of feature importance, and a persisted inference function.

### How to use this notebook
Each section has a short **Context** explaining *why* the step matters, then a **Task** list of exactly what to build. Code cells are stubbed with `# TODO` — replace them with your implementation. Do not look at the Solutions notebook until you've attempted each section. Use the **Cheat Sheet** notebook if you get stuck on syntax, and the **Background Theory** notebook if you get stuck on *why*.

### Success criteria for the whole project
- A cleaned, fully numeric feature matrix with no leakage between train and test
- At least 4 trained regression models, fairly compared on the same test split
- A tuned final model with cross-validated performance estimates
- Two independent feature-importance views that agree on the top price drivers
- A short written interpretation a non-technical stakeholder could read


## Module 0 — Environment Setup

**Context.** Reproducible environments prevent "works on my machine" bugs, and fixing random seeds early means every result you get later is comparable to the solution notebook's.

**Task**
- Import `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`.
- Import `train_test_split`, `cross_val_score`, `KFold` from `sklearn.model_selection`.
- Import `LinearRegression`, `Ridge`, `Lasso` from `sklearn.linear_model`.
- Import `RandomForestRegressor`, `GradientBoostingRegressor` from `sklearn.ensemble`.
- Import `mean_absolute_error`, `mean_squared_error`, `r2_score` from `sklearn.metrics`.
- Set a global `RANDOM_STATE = 42` constant and use it everywhere a `random_state` argument exists.
- Set a seaborn style (e.g. `sns.set_theme(style="whitegrid")`).


In [ ]:
# TODO: imports and global constants


## Module 1 — Data Loading & Initial Inspection

**Context.** Before touching a single value you need to know the shape of the problem: how many rows, how many columns, which columns are already numeric, and which are hiding numbers inside text.

**Task**
- Load `laptop_price.csv` into a DataFrame `df`.
- Print `df.shape`, `df.head()`, and `df.info()`.
- Print `df.describe(include='all').T` to see summary stats for every column, numeric and categorical.
- Write one markdown sentence identifying which columns are numeric-looking but stored as `object`/string dtype.


In [ ]:
# TODO: load data and inspect


## Module 2 — Data Quality Audit

**Context.** Real datasets hide problems that `.info()` alone won't surface: disguised missing values (`"Not Available"`, `"No"`), duplicate rows, and inconsistent category spellings (e.g. `ASUS` vs `acer` casing).

**Task**
- Build a small "dtype audit" DataFrame with one row per column showing: `column`, `current_dtype`, `n_unique`, `sample_values` (first 3 unique values).
- Check `df.duplicated().sum()`.
- For every `object` column, print `df[col].unique()` and flag any values that look like disguised missing data or unit-bearing text.
- Check `df['brand'].unique()` specifically for inconsistent casing (e.g. `acer` vs `ASUS`) and decide whether to normalize it.


In [ ]:
# TODO: dtype audit table


In [ ]:
# TODO: duplicate check + per-column unique value scan


## Module 3 — Feature Engineering I: Memory & Storage

**Context.** `ram_gb`, `ssd`, `hdd`, and `graphic_card_gb` are numeric quantities trapped in strings like `"8 GB"`. You also want composite storage features that reduce dimensionality and add signal a raw ssd/hdd split doesn't capture on its own.

**Task**
- For each of `['ram_gb', 'ssd', 'hdd', 'graphic_card_gb']`, strip `" GB"` and cast to `int`.
- Create `total_storage = ssd + hdd`.
- Create a boolean/int flag `has_ssd = (ssd > 0).astype(int)` and `has_hdd = (hdd > 0).astype(int)`.
- Create `ssd_ratio = ssd / total_storage` (handle the divide-by-zero case where `total_storage == 0`).
- Drop the original `ssd` and `hdd` columns once `total_storage`, `has_ssd`, `has_hdd`, and `ssd_ratio` exist.
- Verify with `df[['ram_gb','total_storage','has_ssd','has_hdd','ssd_ratio']].head()`.


In [ ]:
# TODO: memory & storage feature engineering


## Module 4 — Feature Engineering II: Cleaning OS, Rating & Generation

**Context.** `os_bit`, `rating`, and `processor_gnrtn` need the same "strip text, cast to number" treatment, but each has its own quirks (two singular/plural forms for stars, a `"Not Available"` sentinel for generation).

**Task**
- Convert `os_bit` to numeric (strip `"-bit"`).
- Convert `rating` to numeric (strip both `" stars"` and `" star"`).
- Convert `processor_gnrtn` to numeric, mapping `"Not Available"` to `0` before stripping `"th"`.
- Make every conversion **idempotent**: cast to `str` first so re-running the cell doesn't break on already-clean columns.
- Print `df[['os_bit','rating','processor_gnrtn']].describe()` to sanity check ranges.


In [ ]:
# TODO: clean os_bit, rating, processor_gnrtn


## Module 5 — Feature Engineering III: Domain-Derived Features

**Context.** Raw specs only go so far. A strong tabular-ML entry usually adds a handful of *domain* features that encode expert knowledge the raw columns don't state directly.

**Task** — engineer at least four of the following:
- `processor_tier`: an ordinal score for `processor_name` (e.g. Celeron/Pentium=0, i3/Ryzen3=1, i5/Ryzen5=2, i7/Ryzen7=3, i9/Ryzen9/M1=4).
- `is_premium_brand`: flag for brands like `APPLE`, `MSI` vs. budget brands.
- `popularity_score`: a combination of `Number of Ratings` and `Number of Reviews` (e.g. log1p of their sum).
- `performance_index`: a weighted combination of `ram_gb`, `total_storage`, `graphic_card_gb`, and `processor_tier` (weights are your design choice — document them).
- `is_gaming`: derive from the `weight` column's `"Gaming"` category.
- Justify each new feature in one sentence — what price-relevant signal does it add that raw columns miss?


In [ ]:
# TODO: domain-derived features (at least 4)


## Module 6 — Advanced Exploratory Data Analysis

**Context.** Prices are rarely normally distributed — understanding skew, outliers, and multicollinearity now prevents surprises during modeling.

**Task**
- Plot a histogram + KDE of `Price`. Compute its skewness (`df['Price'].skew()`).
- Create `log_price = np.log1p(df['Price'])` and plot its distribution. Decide (with a written justification) whether to model `Price` or `log_price`.
- Build a correlation heatmap (`numeric_df.corr()`, `annot=True`, `cmap='coolwarm'`) and identify the top 5 features most correlated with `Price`.
- Compute Variance Inflation Factor (VIF) for the numeric features (`statsmodels.stats.outliers_influence.variance_inflation_factor`) and flag any feature with VIF > 10.
- Create boxplots of `Price` grouped by `brand`, `ram_gb`, and `graphic_card_gb`.
- Detect outliers in `Price` using the IQR method; report how many rows would be flagged and decide whether to keep, cap, or remove them (with justification).


In [ ]:
# TODO: target distribution, skew, log transform


In [ ]:
# TODO: correlation heatmap + VIF check


In [ ]:
# TODO: categorical vs price boxplots


In [ ]:
# TODO: IQR-based outlier detection on Price


## Module 7 — Encoding Categorical Variables (Leakage-Safe)

**Context.** Encoding statistics (like a category's mean price) must be learned **only** from the training set, or your test metrics will be optimistic and won't reflect real-world performance.

**Task**
- Identify remaining categorical columns with `df.select_dtypes(include='object').columns`.
- Split columns into low-cardinality (< 5 unique values → one-hot encode with `drop_first=True`) and high-cardinality (≥ 5 unique values → target-encode).
- Perform the train/test split **before** computing any target-encoding statistic.
- Fit target-encoding means using only `X_train`/`y_train`; apply the learned mapping to `X_test`; fill any category unseen in training with the overall training mean.
- Confirm no `NaN` values remain in either `X_train` or `X_test` after encoding.


In [ ]:
# TODO: identify + bucket categorical columns by cardinality


In [ ]:
# TODO: train/test split BEFORE target encoding


In [ ]:
# TODO: fit target encoding on train, apply to test, handle unseen categories


## Module 8 — Baseline Models

**Context.** You cannot claim a model is "good" without a baseline to beat. A mean predictor and a plain linear regression establish the floor.

**Task**
- Build a trivial baseline that always predicts `y_train.mean()`. Compute its MAE/RMSE/R² on the test set.
- Fit a `LinearRegression` on the same features. Compute the same three metrics.
- Fit `Ridge` and `Lasso` with a default alpha. Compare their coefficients to plain linear regression — did regularization zero out or shrink anything meaningfully?


In [ ]:
# TODO: mean-predictor baseline


In [ ]:
# TODO: LinearRegression, Ridge, Lasso


## Module 9 — Tree-Ensemble Models

**Context.** Tree ensembles typically dominate tabular-data leaderboards because they capture non-linear interactions the linear models above cannot.

**Task**
- Train a `RandomForestRegressor(random_state=RANDOM_STATE)` with default hyperparameters.
- Train a `GradientBoostingRegressor(random_state=RANDOM_STATE)` with default hyperparameters.
- (Optional stretch) If `xgboost` is available, train an `XGBRegressor` too.
- Collect MAE/RMSE/R² for every model trained so far in a single comparison table.


In [ ]:
# TODO: RandomForestRegressor


In [ ]:
# TODO: GradientBoostingRegressor (+ optional XGBRegressor)


In [ ]:
# TODO: assemble a model-comparison DataFrame


## Module 10 — Hyperparameter Tuning & Cross-Validation

**Context.** Default hyperparameters are rarely optimal. Cross-validation gives a more robust performance estimate than a single train/test split, and a randomized search efficiently explores a large hyperparameter space.

**Task**
- Run 5-fold cross-validation (`cross_val_score`, scoring `'neg_mean_absolute_error'`) on your best two models from Module 9. Report mean ± std MAE.
- Define a hyperparameter distribution for `RandomForestRegressor` (e.g. `n_estimators`, `max_depth`, `min_samples_split`, `max_features`).
- Run `RandomizedSearchCV` (or `GridSearchCV` if you prefer an exhaustive search over a small grid) with `cv=5` and `scoring='neg_mean_absolute_error'`.
- Report the best hyperparameters and the best cross-validated score.
- Refit the tuned model on the full training set and evaluate it on the held-out test set.


In [ ]:
# TODO: cross_val_score on candidate models


In [ ]:
# TODO: RandomizedSearchCV / GridSearchCV


In [ ]:
# TODO: refit best estimator, evaluate on test set


## Module 11 — Model Evaluation & Diagnostics

**Context.** A single R² number hides a lot. Residual plots and predicted-vs-actual scatterplots reveal *where* a model fails (e.g. underpredicting expensive laptops).

**Task**
- For your final tuned model, compute MAE, RMSE, R², and MAPE (mean absolute percentage error) on the test set.
- Plot predicted vs. actual `Price` as a scatterplot with a `y=x` reference line.
- Plot residuals (`y_test - y_pred`) vs. predicted values; look for heteroscedasticity (a funnel shape).
- Plot a histogram of residuals; check whether they're roughly centered at zero.
- Write two sentences: where does the model struggle most (e.g. very cheap or very expensive laptops)?


In [ ]:
# TODO: MAE, RMSE, R2, MAPE on final model


In [ ]:
# TODO: predicted vs actual scatterplot


In [ ]:
# TODO: residual plots


## Module 12 — Feature Importance

**Context.** Impurity-based importances are fast but biased toward high-cardinality features; permutation importance is slower but measures actual predictive damage on held-out data. Comparing both gives a more trustworthy story.

**Task**
- Extract `feature_importances_` from your final tree-based model and plot the top 10 as a horizontal bar chart.
- Compute permutation importance (`sklearn.inspection.permutation_importance`, `n_repeats=10`) on the test set and plot the top 10.
- Compare the two rankings — do they agree? If not, hypothesize why (cardinality bias is the most likely culprit).


In [ ]:
# TODO: impurity-based feature importance plot


In [ ]:
# TODO: permutation importance plot


## Module 13 — Final Model Persistence & Inference

**Context.** A model that only exists inside a notebook variable isn't useful to anyone else. Persisting it (and the preprocessing needed to use it) makes it deployable.

**Task**
- Save the final tuned model with `joblib.dump`.
- Save any encoding mappings (e.g. target-encoding dictionaries, one-hot column list) needed to transform new raw input the same way.
- Write a function `predict_price(raw_laptop_dict)` that takes a dictionary of raw specs (same shape as one row of the original CSV), applies the full cleaning + feature engineering + encoding pipeline, and returns a predicted price.
- Test your function on 2–3 made-up laptop specs and sanity-check the outputs are in a plausible price range.


In [ ]:
# TODO: persist model + encoders with joblib


In [ ]:
# TODO: predict_price(raw_laptop_dict) function


In [ ]:
# TODO: sanity-check predictions on made-up laptops


## Module 14 — Conclusions & Business Write-Up

**Context.** Technical results only create value once they're translated into a decision a stakeholder can act on.

**Task**
- Write a short (150–250 word) markdown summary covering: which model you chose and why, its expected error margin in real currency, the top 3–5 price drivers, one limitation of the dataset or approach, and one concrete next step (e.g. collect more data on X, try a stacked ensemble, deploy behind an API).


*(Write your conclusions here.)*